# Classificador 

In [1]:
import pandas as pd

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

In [3]:
import numpy as np

In [4]:
from string import punctuation

In [5]:
from sklearn.feature_extraction.text import CountVectorizer

In [38]:
from sklearn.model_selection import train_test_split

In [62]:
import random

In [6]:
df = pd.read_csv("d:/git/dados/nlp/news_sentiment_analysis.csv", encoding="utf-8")

In [7]:
df = df.drop(columns=["Source", "Author", "URL", "Published At", "Sentiment"])

In [37]:
df

,Title,Description,Type,type_number,description_clean
0,Pine View High teacher wins Best in State awar...,"ST. GEORGE — Kaitlyn Larson, a first-year teac...",Business,"[0, 0, 0, 0, 0, 0, 1]",st george — kaitlyn larson a firstyear teacher...
1,Businesses Face Financial Strain Amid Liquidit...,"Harare, Zimbabwe – Local businesses are grappl...",Business,"[0, 0, 0, 0, 0, 0, 1]",harare zimbabwe – local businesses are grappli...
2,Musk donates to super pac working to elect Tru...,(marketscreener.com) Billionaire Elon Musk has...,Business,"[0, 0, 0, 0, 0, 0, 1]",marketscreenercom billionaire elon musk has do...
3,US FTC issues warning to franchisors over unfa...,(marketscreener.com) A U.S. trade regulator on...,Business,"[0, 0, 0, 0, 0, 0, 1]",marketscreenercom a us trade regulator on frid...
4,Rooftop solar's dark side,4.5 million households in the U.S. have solar ...,Business,"[0, 0, 0, 0, 0, 0, 1]",45 million households in the us have solar pan...
...,...,...,...,...,...
3495,"Arrow Electronics, Inc. (NYSE:ARW) Shares Purc...",QRG Capital Management Inc. increased its stak...,Technology,"[1, 0, 0, 0, 0, 0, 0]",qrg capital management inc increased its stake...
3496,"3,120 Shares in NICE Ltd. (NASDAQ:NICE) Bought...",QRG Capital Management Inc. bought a new posit...,Technology,"[1, 0, 0, 0, 0, 0, 0]",qrg capital management inc bought a new positi...
3497,"QRG Capital Management Inc. Has $857,000 Stock...",QRG Capital Management Inc. boosted its stake ...,Technology,"[1, 0, 0, 0, 0, 0, 0]",qrg capital management inc boosted its stake i...
3498,Biotechnology Market: Surging Investments and ...,"WESTFORD, Mass., July 18, 2024 /PRNewswire/ --...",Technology,"[1, 0, 0, 0, 0, 0, 0]",westford mass july 18 2024 prnewswire accordi...


In [9]:
print(df["Type"].unique())
print(df["Type"].describe())

['Business' 'Entertainment' 'General' 'Health' 'Science' 'Sports'
 'Technology']
count         3500
unique           7
top       Business
freq           500
Name: Type, dtype: object


In [10]:
# mapping = {'Business': 1, 'Entertainment': 2, 'General': 3, 'Health': 4, 'Science':5, 'Sports':6, 'Technology': 7}
mapping = {
        'Business': [0, 0, 0, 0, 0, 0, 1], 
        'Entertainment': [0, 0, 0, 0, 0, 1, 0],
        'General': [0, 0, 0, 0, 1, 0, 0],
        'Health': [0, 0, 0, 1, 0, 0, 0], 
        'Science': [0, 0, 1, 0, 0, 0, 0],
        'Sports': [0, 1, 0, 0, 0, 0, 0], 
        'Technology': [1, 0, 0, 0, 0, 0, 0]
}
df["type_number"] = df["Type"].map( mapping ) 

In [11]:
df

,Title,Description,Type,type_number
0,Pine View High teacher wins Best in State awar...,"ST. GEORGE — Kaitlyn Larson, a first-year teac...",Business,"[0, 0, 0, 0, 0, 0, 1]"
1,Businesses Face Financial Strain Amid Liquidit...,"Harare, Zimbabwe – Local businesses are grappl...",Business,"[0, 0, 0, 0, 0, 0, 1]"
2,Musk donates to super pac working to elect Tru...,(marketscreener.com) Billionaire Elon Musk has...,Business,"[0, 0, 0, 0, 0, 0, 1]"
3,US FTC issues warning to franchisors over unfa...,(marketscreener.com) A U.S. trade regulator on...,Business,"[0, 0, 0, 0, 0, 0, 1]"
4,Rooftop solar's dark side,4.5 million households in the U.S. have solar ...,Business,"[0, 0, 0, 0, 0, 0, 1]"
...,...,...,...,...
3495,"Arrow Electronics, Inc. (NYSE:ARW) Shares Purc...",QRG Capital Management Inc. increased its stak...,Technology,"[1, 0, 0, 0, 0, 0, 0]"
3496,"3,120 Shares in NICE Ltd. (NASDAQ:NICE) Bought...",QRG Capital Management Inc. bought a new posit...,Technology,"[1, 0, 0, 0, 0, 0, 0]"
3497,"QRG Capital Management Inc. Has $857,000 Stock...",QRG Capital Management Inc. boosted its stake ...,Technology,"[1, 0, 0, 0, 0, 0, 0]"
3498,Biotechnology Market: Surging Investments and ...,"WESTFORD, Mass., July 18, 2024 /PRNewswire/ --...",Technology,"[1, 0, 0, 0, 0, 0, 0]"


In [12]:
Y = torch.tensor(df["type_number"], dtype=torch.float32)
Y = torch.reshape(Y, (-1, 7))

In [13]:
Y.shape

torch.Size([3500, 7])

In [126]:
table = str.maketrans("", "", punctuation)

# def limpar( texto ):
#     texto_limpo = texto.lower().translate(table)
#     return texto_limpo

def limpar( texto ):
    return texto

In [130]:
df["description_clean"] = df["Description"].apply(limpar)

In [131]:
MAX_PALAVRAS = 500

In [132]:
vetorizador = CountVectorizer(max_features = MAX_PALAVRAS)

In [133]:
texto_vetorizado = vetorizador.fit_transform(df["description_clean"])

In [150]:
dicionario = vetorizador.get_feature_names_out()
dicionario

array(['00', '000', '10', '11', '12', '13', '13f', '14', '15', '16',
       '160', '17', '18', '1st', '20', '2020', '2023', '2024', '20240712',
       '20240716', '20240718', '24', '25', '30', '39', '8211', '8212',
       '8216', '8217', '8220', '8221', '8230', '92nd', 'about',
       'according', 'acquired', 'across', 'action', 'additional', 'after',
       'against', 'agency', 'ai', 'air', 'airman', 'al', 'all', 'also',
       'america', 'american', 'among', 'an', 'analysts', 'and',
       'announced', 'announces', 'annual', 'ap', 'appeared',
       'approximately', 'are', 'army', 'around', 'art', 'as', 'assigned',
       'at', 'attention', 'attorney', 'auf', 'august', 'average', 'back',
       'bank', 'base', 'based', 'be', 'been', 'before', 'bei', 'being',
       'best', 'between', 'biden', 'big', 'board', 'both', 'bought',
       'brown', 'business', 'businesses', 'but', 'by', 'can', 'canada',
       'canadian', 'capabilities', 'capital', 'care', 'center', 'channel',
       'che',

In [148]:
X = torch.tensor(texto_vetorizado.toarray(), dtype=torch.float32)
X

tensor([[0., 0., 1.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 1.,  ..., 0., 0., 0.]])

In [137]:
print("X: ", X.dtype, X.shape, X.ndim)
print("Y: ", Y.dtype, Y.shape, Y.ndim)

X:  torch.float32 torch.Size([3500, 500]) 2
Y:  torch.float32 torch.Size([3500, 7]) 2


In [138]:
X_treino, X_teste, Y_treino, Y_teste = train_test_split(X, Y, test_size=0.2, random_state=100)

In [139]:
len(X_treino)
# X_treino[0].sum()

2800

In [140]:
modelo = nn.Linear(in_features = MAX_PALAVRAS, out_features=7)

In [141]:
criterio = nn.CrossEntropyLoss() # Cross Entropy
otimizador = optim.SGD( modelo.parameters(), lr=0.01 )

In [142]:
for epoca in range(1, 1000):
    Y_hat = modelo( X_treino )
    loss = criterio( Y_hat, Y_treino )
    otimizador.zero_grad()
    loss.backward()
    otimizador.step()
    if epoca % 100 == 0:
        print(f"Epoca: {epoca}\tLoss:{loss}")

Epoca: 100	Loss:1.7638030052185059
Epoca: 200	Loss:1.6200628280639648
Epoca: 300	Loss:1.505477786064148
Epoca: 400	Loss:1.4099808931350708
Epoca: 500	Loss:1.328412652015686
Epoca: 600	Loss:1.2576364278793335
Epoca: 700	Loss:1.1955220699310303
Epoca: 800	Loss:1.1405165195465088
Epoca: 900	Loss:1.0914374589920044


In [143]:
vetorizador_predict = CountVectorizer(max_features = MAX_PALAVRAS, vocabulary=dicionario)

In [144]:
indice = random.randint(0, 3500)
predict_set = [ df["description_clean"][indice] ]
tipo = [ df["Type"][indice] ]
print(f"Indice: {indice}\t\tTipo: {tipo}")
print(predict_set)

# predict_set = [
#     # "the fruitwatch initiative a groundbreaking citizen science project has significantly enhanced the accuracy of predicting flowering times for fruit trees across great britain this improvement is vital for the agricultural sector enabling better planning for pest management and pollinator support which are crucial for maintaining optimal fruit yield and quality"
#     # "researchers at the institute for systems biology in seattle found that bowel movement frequency could predict kidney and liver damage as well as mental health issues like depression"
#     # "ap entertainment writer new york ap — richard simmons television8217s hyperactive court jester of physical fitness who built a miniempire in his trademark tank tops and short shorts by urging the overweight to exercise and eat better died saturday he turned 76 on friday los angeles police and fire departments say they responded to athe post richard simmons a fitness guru who mixed laughs and sweat dies at 76 appeared first on kvia"
# ]

Indice: 2901		Tipo: ['Entertainment']
['Image: Blizzard Entertainment The Spiritborn comes with the Diablo 4: Vessel of Hatred expansion on Oct. 8 Continue reading&hellip;']


In [145]:
X_predict = vetorizador_predict.fit_transform( predict_set )
list_reverse = ['Technology', 'Sports', 'Science', 'Health', 'General', 'Entertainment', 'Business']
with torch.no_grad():
    X_pred = torch.tensor(X_predict.toarray(), dtype=torch.float32)
    resposta = modelo( X_pred )
    indice = np.argmax( resposta )
    print(f"Este texto é sobre {list_reverse[indice]}")
# X_predict.toarray()

Este texto é sobre Entertainment


In [149]:
X_pred

tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0.,

In [146]:
modelo.eval()

with torch.no_grad():
    vrespostas = modelo( X_teste )
    max_respostas = np.argmax( vrespostas, axis=1 )
    max_Y_teste = np.argmax(Y_teste, axis=1)

In [90]:
# print(max_respostas)

In [89]:
# print(max_Y_teste)

In [147]:
resultados = (max_respostas - max_Y_teste)
total = len(resultados)
total_erros = len(resultados[ resultados > 0 ])
total_acertos = total - total_erros

acerto = total_acertos / total
print("Acerto do modelo ==>", acerto)

Acerto do modelo ==> 0.9128571428571428


Com 500 palavras o Acerto do modelo e texto limpo ==> 0.8957142857142857
Com 500 palavras o Acerto do modelo e texto sem limpar ==>  0.9128571428571428
Com 1000 palavras o Acerto do modelo e texto limpo  ==> 0.9057142857142857
Com 2000 palavras o Acerto do modelo e texto limpo  ==> 0.8971428571428571

In [36]:
# mapping = {
#         'Business': [0, 0, 0, 0, 0, 0, 1], 
#         'Entertainment': [0, 0, 0, 0, 0, 1, 0],
#         'General': [0, 0, 0, 0, 1, 0, 0],
#         'Health': [0, 0, 0, 1, 0, 0, 0], 
#         'Science': [0, 0, 1, 0, 0, 0, 0],
#         'Sports': [0, 1, 0, 0, 0, 0, 0], 
#         'Technology': [1, 0, 0, 0, 0, 0, 0]
# }
# list_reverse = ['Technology', 'Sports', 'Science', 'Health', 'General', 'Entertainment', 'Business']